In [0]:
# =============================================================================
# Smart ERP – Product Silver Layer Simulation  (Databricks / Delta Lake)
# Catalog  : smart_odoo
# Targets  :
#     silver.product_category   (10 rows)
#     silver.product_template   (100 rows : 70 laptops + 30 accessories)
#     silver.product_product    (100 rows : 1-to-1 with template)
#
# IDEMPOTENCY GUARANTEES
# ──────────────────────
# ✅ Fixed seed          → identical names / prices / dates on every run
# ✅ Deterministic picks → no random.choice on accessories (index-based)
# ✅ MERGE on PK         → re-running never creates duplicate rows
# ✅ Table created once  → saveAsTable("errorifexists") on first run only
# ✅ dwh_loaded_at       → refreshed to current_timestamp() on every upsert
# ✅ Temp-views dropped  → no stale-view collision across runs
# ✅ Pre-flight asserts  → catches logic errors before touching Delta
#
# Product rules
# ─────────────
#   product_id  1 – 70   → Laptops     (type = "Laptop")
#   product_id 71 – 100  → Accessories (type = "Accessory")
#   Date range           : 2025-01-01 → 2026-04-22
# =============================================================================

# ── 0. Imports ────────────────────────────────────────────────────────────────
import random
import string
from datetime import datetime, timedelta, timezone

from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, StringType, BooleanType,
    DoubleType, TimestampType,
)

# ── Fixed seeds → same output on every run ────────────────────────────────────
SEED = 42
random.seed(SEED)

spark = SparkSession.builder.getOrCreate()

# ── 1. Constants ──────────────────────────────────────────────────────────────
CATALOG   = "smart_odoo"
SCHEMA    = "silver"
SOURCE_DB = "python_ingest"

START_DATE = datetime(2025, 1, 1)
END_DATE   = datetime(2026, 4, 22)

LAPTOPS = [
    ("Dell",   "XPS",        ["13 9310", "15 9510"],   ["i5","i7","i9"], ["11th Gen","12th Gen"]),
    ("HP",     "ZBook Fury", ["16 G9",   "15 G8"],     ["i7","i9"],      ["11th Gen","12th Gen"]),
    ("Lenovo", "ThinkPad T", ["14 Gen 2","14 Gen 3"],  ["i5","i7"],      ["11th Gen","12th Gen"]),
    ("Apple",  "MacBook Pro",["14-inch", "16-inch"],   ["M2","M3"],      ["Base"]),
]

ACCESSORIES = [
    "Logitech MX Master 3S Mouse",
    "Logitech MX Keys Keyboard",
    "Razer DeathAdder V2 Mouse",
    "Anker USB-C Hub 7-in-1",
    "External SSD 1TB Portable",
]

BRAND_CATEG = {"Dell": 4, "HP": 5, "Lenovo": 6, "Apple": 7}

# ── 2. Helpers ────────────────────────────────────────────────────────────────
def _rdate() -> datetime:
    return START_DATE + timedelta(
        days=random.randint(0, (END_DATE - START_DATE).days)
    )

def _laptop_price(cpu: str) -> float:
    base = {"i5": 25000, "i7": 35000, "i9": 50000, "M2": 50000, "M3": 65000}
    return float(base.get(cpu, 30000) + random.randint(-2000, 5000))

def _acc_price(name: str) -> float:
    if "Mouse"    in name: return float(random.randint(800,  2500))
    if "Keyboard" in name: return float(random.randint(1200, 4000))
    return float(random.randint(1500, 6000))

def _cost(price: float) -> float:
    return round(price * random.uniform(0.85, 0.95), 2)

def _sku(brand: str, series: str, model: str, cpu: str) -> str:
    return (f"{brand[:3].upper()}-{series[:3].upper()}"
            f"-{model.replace(' ','')}-{cpu.replace(' ','')}")

def _acc_categ(name: str) -> int:
    if "Mouse"    in name: return 8
    if "Keyboard" in name: return 9
    return 10

# ── 3. Schemas ────────────────────────────────────────────────────────────────
categ_schema = StructType([
    StructField("categ_id",                          IntegerType(),  False),
    StructField("parent_id",                         IntegerType(),  True),
    StructField("parent_name",                       StringType(),   True),
    StructField("name",                              StringType(),   True),
    StructField("complete_name",                     StringType(),   True),
    StructField("parent_path",                       StringType(),   True),
    StructField("product_properties_definition",     StringType(),   True),
    StructField("property_account_income_categ_id",  IntegerType(),  True),
    StructField("property_account_income_categ_name",StringType(),   True),
    StructField("property_account_expense_categ_id", IntegerType(),  True),
    StructField("property_account_expense_categ_name",StringType(),  True),
    StructField("created_at",                        TimestampType(), True),
    StructField("updated_at",                        TimestampType(), True),
    StructField("dwh_loaded_at",                     TimestampType(), True),
    StructField("dwh_source_db",                     StringType(),   True),
])

tmpl_schema = StructType([
    StructField("product_tmpl_id",              IntegerType(),  False),
    StructField("categ_id",                     IntegerType(),  True),
    StructField("categ_name",                   StringType(),   True),
    StructField("uom_id",                       IntegerType(),  True),
    StructField("uom_name",                     StringType(),   True),
    StructField("company_id",                   IntegerType(),  True),
    StructField("company_name",                 StringType(),   True),
    StructField("color",                        IntegerType(),  True),
    StructField("type",                         StringType(),   True),
    StructField("service_tracking",             StringType(),   True),
    StructField("default_code",                 StringType(),   True),
    StructField("name",                         StringType(),   True),
    StructField("description",                  StringType(),   True),
    StructField("description_purchase",         StringType(),   True),
    StructField("description_sale",             StringType(),   True),
    StructField("product_properties",           StringType(),   True),
    StructField("list_price",                   DoubleType(),   True),
    StructField("volume",                       DoubleType(),   True),
    StructField("weight",                       DoubleType(),   True),
    StructField("sale_ok",                      BooleanType(),  True),
    StructField("purchase_ok",                  BooleanType(),  True),
    StructField("active",                       BooleanType(),  True),
    StructField("can_image_1024_be_zoomed",     BooleanType(),  True),
    StructField("has_configurable_attributes",  BooleanType(),  True),
    StructField("is_favorite",                  BooleanType(),  True),
    StructField("property_account_income_id",   IntegerType(),  True),
    StructField("property_account_income_name", StringType(),   True),
    StructField("property_account_expense_id",  IntegerType(),  True),
    StructField("property_account_expense_name",StringType(),   True),
    StructField("service_type",                 StringType(),   True),
    StructField("expense_policy",               StringType(),   True),
    StructField("invoice_policy",               StringType(),   True),
    StructField("sale_line_warn_msg",           StringType(),   True),
    StructField("created_at",                   TimestampType(), True),
    StructField("updated_at",                   TimestampType(), True),
    StructField("dwh_loaded_at",                TimestampType(), True),
    StructField("dwh_source_db",                StringType(),   True),
])

product_schema = StructType([
    StructField("product_id",                       IntegerType(),  False),
    StructField("product_tmpl_id",                  IntegerType(),  True),
    StructField("product_tmpl_name",                StringType(),   True),
    StructField("default_code",                     StringType(),   True),
    StructField("barcode",                          StringType(),   True),
    StructField("combination_indices",              StringType(),   True),
    StructField("standard_price",                   DoubleType(),   True),
    StructField("volume",                           DoubleType(),   True),
    StructField("weight",                           DoubleType(),   True),
    StructField("active",                           BooleanType(),  True),
    StructField("can_image_variant_1024_be_zoomed", BooleanType(),  True),
    StructField("is_favorite",                      BooleanType(),  True),
    StructField("is_in_selected_section_of_order",  BooleanType(),  True),
    StructField("created_at",                       TimestampType(), True),
    StructField("updated_at",                       TimestampType(), True),
    StructField("dwh_loaded_at",                    TimestampType(), True),
    StructField("dwh_source_db",                    StringType(),   True),
])

# ── 4. Build product_category rows ───────────────────────────────────────────
#  (categ_id, parent_id, parent_name, name, complete_name, parent_path)
CATEGORIES_RAW = [
    (1,  None, None,           "All Products", "All Products",                         "1/"),
    (2,  1,    "All Products", "Laptops",      "All Products / Laptops",               "1/2/"),
    (3,  1,    "All Products", "Accessories",  "All Products / Accessories",           "1/3/"),
    (4,  2,    "Laptops",      "Dell",         "All Products / Laptops / Dell",        "1/2/4/"),
    (5,  2,    "Laptops",      "HP",           "All Products / Laptops / HP",          "1/2/5/"),
    (6,  2,    "Laptops",      "Lenovo",       "All Products / Laptops / Lenovo",      "1/2/6/"),
    (7,  2,    "Laptops",      "Apple",        "All Products / Laptops / Apple",       "1/2/7/"),
    (8,  3,    "Accessories",  "Mouse",        "All Products / Accessories / Mouse",   "1/3/8/"),
    (9,  3,    "Accessories",  "Keyboard",     "All Products / Accessories / Keyboard","1/3/9/"),
    (10, 3,    "Accessories",  "Other",        "All Products / Accessories / Other",   "1/3/10/"),
]

now = datetime.now(timezone.utc).replace(tzinfo=None)   # naive UTC for Delta

categ_rows = []
for cid, pid, pname, name, cname, path in CATEGORIES_RAW:
    categ_rows.append((
        cid, pid, pname, name, cname, path,
        None,           # product_properties_definition
        None, None,     # property_account_income_categ_id / name
        None, None,     # property_account_expense_categ_id / name
        now, now, now, SOURCE_DB,
    ))

# Build categ_id → complete_name lookup for template categ_name column
categ_name_by_id = {r[0]: r[4] for r in categ_rows}

# ── 5. Build product_template + product_product rows ─────────────────────────
tmpl_rows    = []
product_rows = []

# ── 5a. Laptops (product_id 1-70) ─────────────────────────────────────────────
for pid in range(1, 71):
    brand, series, models, cpus, gens = random.choice(LAPTOPS)
    model   = random.choice(models)
    cpu     = random.choice(cpus)
    gen     = random.choice(gens)
    name    = (f"{brand} {series} {model} {cpu}"
               if brand == "Apple"
               else f"{brand} {series} {model} {cpu} {gen}")
    price   = _laptop_price(cpu)
    cost    = _cost(price)
    weight  = round(random.uniform(1.2, 2.8), 2)
    sku     = _sku(brand, series, model, cpu)
    catid   = BRAND_CATEG[brand]
    created = _rdate()

    tmpl_rows.append((
        pid,
        catid,
        categ_name_by_id[catid],    # categ_name  (full path)
        1,  "Unit(s)",              # uom_id / uom_name
        1,  "Alpha Corp",           # company_id / company_name
        0,                          # color
        "Laptop",                   # type
        None,                       # service_tracking
        sku,                        # default_code
        name,                       # name
        None, None, None, None,     # description fields
        price,                      # list_price
        0.0,                        # volume
        weight,                     # weight
        True, True, True,           # sale_ok, purchase_ok, active
        False, False, False,        # can_image, has_configurable, is_favorite
        None, None, None, None,     # account income/expense
        None, None, None, None,     # service_type, expense_policy, invoice_policy, warn
        created, created, now, SOURCE_DB,
    ))
    product_rows.append((
        pid,                        # product_id
        pid,                        # product_tmpl_id
        name,                       # product_tmpl_name
        sku,                        # default_code
        None,                       # barcode
        None,                       # combination_indices
        cost,                       # standard_price
        0.0,                        # volume
        weight,                     # weight
        True,                       # active
        False,                      # can_image_variant_1024_be_zoomed
        False,                      # is_favorite
        False,                      # is_in_selected_section_of_order
        created, created, now, SOURCE_DB,
    ))

# ── 5b. Accessories (product_id 71-100) ───────────────────────────────────────
for pid in range(71, 101):
    # Deterministic pick (no random.choice) → same name every run
    name    = ACCESSORIES[(pid - 71) % len(ACCESSORIES)]
    price   = _acc_price(name)
    cost    = _cost(price)
    weight  = round(random.uniform(0.2, 1.5), 2)
    sku     = "ACC-" + name.split(" ")[0].upper()
    catid   = _acc_categ(name)
    created = _rdate()

    tmpl_rows.append((
        pid,
        catid,
        categ_name_by_id[catid],
        1, "Unit(s)",
        1, "Alpha Corp",
        0,
        "Accessory",
        None,
        sku, name,
        None, None, None, None,
        price,
        0.0, weight,
        True, True, True,
        False, False, False,
        None, None, None, None,
        None, None, None, None,
        created, created, now, SOURCE_DB,
    ))
    product_rows.append((
        pid, pid, name, sku,
        None, None,
        cost,
        0.0, weight,
        True, False, False, False,
        created, created, now, SOURCE_DB,
    ))

# ── 6. Create Spark DataFrames ────────────────────────────────────────────────
df_categ   = spark.createDataFrame(categ_rows,   schema=categ_schema)
df_template = spark.createDataFrame(tmpl_rows,   schema=tmpl_schema)
df_product  = spark.createDataFrame(product_rows, schema=product_schema)

# ── 7. Pre-flight assertions ──────────────────────────────────────────────────
def _preflight():
    cids = [r[0] for r in categ_rows]
    tids = [r[0] for r in tmpl_rows]
    pids = [r[0] for r in product_rows]

    assert len(cids) == len(set(cids)),  "❌ Duplicate categ_id"
    assert len(tids) == len(set(tids)),  "❌ Duplicate product_tmpl_id"
    assert len(pids) == len(set(pids)),  "❌ Duplicate product_id"

    assert len(categ_rows)   == 10,  f"❌ Expected 10 categories, got {len(categ_rows)}"
    assert len(tmpl_rows)    == 100, f"❌ Expected 100 templates,  got {len(tmpl_rows)}"
    assert len(product_rows) == 100, f"❌ Expected 100 products,   got {len(product_rows)}"

    laptops_t = [r for r in tmpl_rows if r[0] <= 70]
    acc_t     = [r for r in tmpl_rows if r[0] >= 71]
    assert all(r[8] == "Laptop"    for r in laptops_t), "❌ Type mismatch in laptops"
    assert all(r[8] == "Accessory" for r in acc_t),     "❌ Type mismatch in accessories"

    for r in tmpl_rows:
        d = r[-4]   # created_at position
        assert START_DATE <= d <= END_DATE, f"❌ Date out of range: {d}"

    print("✅  Pre-flight assertions passed")

_preflight()

# ── 8. Idempotent MERGE writer ────────────────────────────────────────────────
def upsert_silver(df, table: str, pk: str) -> None:
    """
    Safely writes df to smart_odoo.silver.<table>.

    Run 1  → creates the Delta table via saveAsTable (errorifexists).
    Run 2+ → MERGE on <pk>:
               MATCHED     → update all columns; dwh_loaded_at = current_timestamp()
               NOT MATCHED → insert new row
    Temp view is dropped after each call to keep the session clean.
    """
    full_table = f"{CATALOG}.{SCHEMA}.{table}"
    view_name  = f"_sim_src_{table}"            # unique per table, no collision

    table_exists = spark.catalog.tableExists(full_table)

    if not table_exists:
        # ── First run: create the table ───────────────────────────────────────
        (df.write
           .format("delta")
           .mode("errorifexists")               # hard-fail on race condition
           .option("overwriteSchema", "false")
           .saveAsTable(full_table))
        print(f"  [created]  {full_table}  →  {df.count()} rows")
        return

    # ── Subsequent runs: MERGE on PK ─────────────────────────────────────────
    df.createOrReplaceTempView(view_name)

    # All columns except PK and dwh_loaded_at (handled via current_timestamp)
    other_cols = [c for c in df.columns if c not in (pk, "dwh_loaded_at")]
    set_parts  = [f"t.{c} = s.{c}" for c in other_cols]
    set_parts.append("t.dwh_loaded_at = current_timestamp()")
    set_clause = ",\n            ".join(set_parts)

    spark.sql(f"""
        MERGE INTO {full_table} AS t
        USING {view_name}       AS s
        ON t.{pk} = s.{pk}

        WHEN MATCHED THEN UPDATE SET
            {set_clause}

        WHEN NOT MATCHED THEN INSERT *
    """)

    spark.catalog.dropTempView(view_name)       # clean up immediately
    print(f"  [merged]   {full_table}  →  upserted {df.count()} rows")


# ── 9. Execute ────────────────────────────────────────────────────────────────
spark.sql(f"USE CATALOG {CATALOG}")

# Order matters: category first (FK dependency)
print("\nWriting silver layer …")
upsert_silver(df_categ,    "product_category", "categ_id")
upsert_silver(df_template, "product_template", "product_tmpl_id")
upsert_silver(df_product,  "product_product",  "product_id")

print(f"""
✅Product simulation complete – smart_odoo.silver populated.
   product_category : {df_categ.count()}  rows
   product_template : {df_template.count()} rows  (laptops=70 | accessories=30)
   product_product  : {df_product.count()} rows
""")